# Second-life garment routing

Attach the private dataset `madmods96/second-life-garment-routing`, turn on Internet, and use a GPU.

The repository is private. Add a GitHub personal access token as a Kaggle secret named `GITHUB_TOKEN` (contents read is enough). The first cell uses that secret to clone the package and does not replace the CUDA build of PyTorch already on Kaggle. Training then runs `configs/front_only.yaml` and `configs/baseline.yaml`. Checkpoints go to `/kaggle/working/runs` and the result files go to `/kaggle/working`.

Create the dataset from the `data/kaggle` folder. The Kaggle CLI keeps a new dataset private unless `--public` is passed:

```bash
kaggle datasets create -p data/kaggle
```

If the Kaggle username is not `madmods96`, change the `id` in `dataset-metadata.json` before that command.


In [ ]:
import os
import stat
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
os.environ["GIT_TERMINAL_PROMPT"] = "0"
askpass = Path("/tmp/git-askpass.sh")
askpass.write_text(
    "#!/bin/sh\n"
    "case \"$1\" in\n"
    "  Username*) echo x-access-token ;;\n"
    "  *) echo \"$GITHUB_TOKEN\" ;;\n"
    "esac\n"
)
askpass.chmod(askpass.stat().st_mode | stat.S_IEXEC)
os.environ["GIT_ASKPASS"] = str(askpass)
REPO = "/tmp/garment-routing"
subprocess.check_call(
    ["git", "clone", "--depth", "1", "https://github.com/MadMods96/garment-routing.git", REPO]
)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "timm", "lightgbm", "pyarrow", "imagehash"]
)


In [ ]:
from pathlib import Path

import pandas as pd

SLUG = "second-life-garment-routing"
candidates = [
    Path("/kaggle/input") / SLUG,
    Path("/kaggle/input/datasets/madmods96") / SLUG,
]
root = next((path for path in candidates if (path / "index.parquet").is_file()), None)
if root is None:
    matches = list(Path("/kaggle/input").glob("**/index.parquet"))
    if len(matches) != 1:
        raise FileNotFoundError(matches)
    root = matches[0].parent

index_path = root / "index.parquet"
image_root = root / "resized"
frame = pd.read_parquet(index_path)
print(root)
print(frame["split"].value_counts().to_string())
print("jpeg files", sum(1 for _ in image_root.rglob("*.jpg")))


In [ ]:
from pathlib import Path

from garmentroute.evaluate import main as evaluate_main
from garmentroute.train import train
from garmentroute.util import load_yaml

REPO = Path("/tmp/garment-routing")
WORKING = Path("/kaggle/working")


def run_named(config_name: str) -> Path:
    config = load_yaml(REPO / "configs" / config_name)
    config["index"] = str(index_path)
    config["image_root"] = str(image_root)
    config["run_dir"] = str(WORKING / "runs")
    return train(config, REPO / "configs" / config_name)


def score(checkpoint: Path, out_name: str, mc_passes: int) -> None:
    evaluate_main(
        [
            "--index", str(index_path),
            "--image-root", str(image_root),
            "--router-config", str(REPO / "configs" / "router.yaml"),
            "--attribute-config", str(REPO / "configs" / "attribute_baseline.yaml"),
            "--checkpoint", str(checkpoint),
            "--out", str(WORKING / out_name),
            "--batch-size", "32",
            "--mc-passes", str(mc_passes),
        ]
    )


front_checkpoint = run_named("front_only.yaml")
score(front_checkpoint, "results_front.json", 0)
baseline_checkpoint = run_named("baseline.yaml")
baseline_passes = int(load_yaml(REPO / "configs" / "baseline.yaml").get("mc_passes", 0))
score(baseline_checkpoint, "results_baseline.json", baseline_passes)
print(front_checkpoint)
print(baseline_checkpoint)
